> Download and tidy up *real* data for training.

In [1]:
import pathlib
import os

import torch
import pandas as pd

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.data
import trapallada.plot

import altair as alt

So that the notebook is aware of modifications in the the `import`ed modules

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory where observations' data will be saved

In [3]:
download_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
download_directory.mkdir(exist_ok=True)

## Data for Javi's paper

In [4]:
obs = '0605950101' # Javi's paper
obs_dir = trapallada.xsa.access.download_extract_filter(obs=obs, output_directory=download_directory)
obs_dir

found directory "/home/manu/data/trapallada/0605950101", skipping...


PosixPath('/home/manu/data/trapallada/0605950101')

The downloaded files

In [5]:
# list(obs_dir.iterdir())
fits_files = list(obs_dir.glob('*.FTZ'))
fits_files[:3]

[PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC000C.FTZ'),
 PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR802C.FTZ'),
 PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR8016.FTZ')]

In [6]:
df, skipped_files_list = trapallada.xsa.util.read_fits_files(fits_files)
print(f'len{skipped_files_list=}')
df.head()

lenskipped_files_list=[PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC000C.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0014.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0015.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC001D.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0020.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0001.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC002B.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0010.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0021.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC0003.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC002A.FTZ'), PosixPath('/home/manu/data/trapallada/0605950101/P060

,P0605950101PNS003SRCTSR802C.FTZ,P0605950101PNS003SRCTSR8016.FTZ,P0605950101PNS003SRCTSR801F.FTZ,P0605950101PNX000SRCTSR8019.FTZ,P0605950101PNX000SRCTSR8009.FTZ,P0605950101PNS003SRCTSR8020.FTZ,P0605950101PNS003SRCTSR800F.FTZ,P0605950101PNX000SRCTSR801A.FTZ,P0605950101PNX000SRCTSR801C.FTZ,P0605950101PNX000SRCTSR8012.FTZ,...,P0605950101PNX000SRCTSR802A.FTZ,P0605950101PNX000SRCTSR8001.FTZ,P0605950101PNX000SRCTSR8003.FTZ,P0605950101PNX000SRCTSR804C.FTZ,P0605950101PNS003SRCTSR801A.FTZ,P0605950101PNX000SRCTSR8008.FTZ,P0605950101PNS003SRCTSR8009.FTZ,P0605950101PNS003SRCTSR8002.FTZ,P0605950101PNS003SRCTSR8032.FTZ,P0605950101PNX000SRCTSR8029.FTZ
TIME,,,,,,,,,,,,,,,,,,,,,
3.604039e+08,0.616343,0.709275,0.736227,-0.073827,3.649674,-0.069906,-0.632557,-0.069920,-0.094558,0.696522,...,-0.104381,34.577186,7.729602,-0.228722,-0.069920,5.138829,3.649674,46.343955,0.698373,1.394028
3.604040e+08,-0.052528,0.853867,-0.132543,-0.167527,1.150647,0.686867,2.253208,0.698083,-0.094227,-0.106176,...,-0.112227,28.529714,4.943364,-0.094323,0.698083,9.597126,1.150647,36.475717,-0.173611,1.336584
3.604041e+08,0.659836,0.703072,-0.252387,-0.118099,-0.346302,1.459107,0.575144,-0.171560,-0.107213,0.684639,...,-0.207021,48.805367,14.755459,-0.188976,-0.171560,7.259375,-0.346302,25.931725,0.522789,0.605379
3.604042e+08,1.515753,0.724594,-0.127593,-0.083033,-0.829979,-0.100040,0.359564,0.877948,-0.106646,0.000000,...,-0.138336,39.296789,12.232534,0.640891,0.877948,3.816259,-0.829979,39.233786,-0.149306,0.532102
3.604043e+08,-0.082870,2.232283,-0.024802,-0.145718,0.204218,-0.070155,0.418255,-0.155570,-0.060782,-0.106507,...,-0.069356,36.757868,13.566273,-0.343177,-0.155570,3.294879,0.204218,31.612971,-0.172412,1.415088


In [7]:
df.shape

(504, 98)

The period is

In [9]:
df.index[1] - df.index[0]

91.97999995946884

Signals might have `NaN`s

In [ ]:
signals = df.values.T
np.isnan(signals).sum()
# np.argwhere(np.isnan(signals))

if there are `NaN`s, they are fixed by interpolation

In [ ]:
if df.isna().any().any():
    df = df.interpolate(method='linear', axis=0)

Output file...

In [ ]:
output_directory = pathlib.Path.cwd().parent / 'data'
assert output_directory.exists()

In [ ]:
# output_file = (output_directory / f'{obs}').with_suffix('.parquet')
output_file = (output_directory / f'{obs}').with_suffix('.pt')
output_file

...to save the signals (notice we add an extra *dummy* axis)

In [ ]:
# df.to_parquet(output_file)
torch.save(np.expand_dims(df.values.T, 1), output_file)

A `SignalsDataset` is instantiated

In [ ]:
ds = trapallada.data.SignalsDataset(output_file, metadata_filename=None, normalize=True)
len(ds)

A `DataFrame` is built out of the `Dataset` in order to make the plot (notice columns are renamed to avoid `.`s). `trapallada.plot.signals_dataset` cannot be called since this is not a *fullsome* `SignalsDataset`(no *metadata* `DataFrame`).

In [ ]:
trapallada.plot.signals_dataset(ds, columns=[pathlib.Path(c).stem for c in df.columns], n_signals=None)
# chart_df = pd.DataFrame(np.squeeze(ds.signals).T, columns=[pathlib.Path(c).stem for c in df.columns]).iloc[:, :]
# chart = alt.Chart(
#     chart_df.reset_index(names='index')
# ).transform_fold(
#     chart_df.columns.to_list()
# ).mark_line().encode(
#     x='index:T', y='value:Q', color='key:N'
# ).properties(width=1_200, height=400)
# chart

In [ ]:
sample_signal = df.iloc[:, 0].values
trapallada.plot.signal(sample_signal)